In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='others/Morgan_ablation/Morgan_test')
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'others/Morgan_ablation/Morgan_test')



In [ ]:
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.metrics import precision_recall_curve,log_loss,accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,matthews_corrcoef,roc_auc_score
import optuna
import gc
from scipy.stats import norm
from collections import Counter
from scipy.stats import norm
from functools import partial
from sklearn.model_selection import StratifiedKFold
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

In [ ]:
# Bin bioactivity values
def bio_cal(bioactivity):
    # PKIS1 contains values above 100 and below 0; clip them before binning
    bioactivity = bioactivity.clip(0,100)
    # Define fixed bin intervals and their labels
    bins = [-np.inf, 50, np.inf]  # Use left-closed, right-open intervals to classify logit-normalized values
    labels = [0, 1]  # Bin labels represent normalized activity levels
    
    final_bioactivity = pd.cut(
    bioactivity, 
    bins=bins, 
    labels=labels, 
    include_lowest=True
    )
    return final_bioactivity

In [ ]:
warm_train = pd.read_csv(_kp('dataset/train_ready2use_merged_4datasets_deduplicated_with_smiles.csv'))

warm_klifs = warm_train['KLIFS']

warm_bio = warm_train['unified_bio_label']

warm_smiles = warm_train['SMILES']

KLIFS_sae_top = pd.read_csv(_kp('feature_embedding/Kinase_test_MCA/MCA_klifs_482_200D_features.csv'),skiprows=1, header=None).values
KLIFS_file = pd.read_csv(_kp('feature_embedding/Kinase_test_MCA/human_482_typical_kinase_1955bit.csv'))

In [ ]:
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)

In [ ]:
# Generate a NumPy fingerprint matrix directly to avoid DataFrame memory overhead
valid_smiles = []
fp_array = np.zeros((len(warm_smiles), 4096), dtype=np.uint8)

valid_idx = 0

for smi in warm_smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print(f"[Warning] Unable to parse SMILES: {smi}")
        continue

    fp = mfpgen.GetFingerprint(mol)
    ConvertToNumpyArray(fp, fp_array[valid_idx])
    valid_smiles.append(smi)
    valid_idx += 1

fp_array = fp_array[:valid_idx]

# Do not create warm_train_mol_fp = pd.DataFrame(...)
print("Morgan fingerprint shape:", fp_array.shape)
print("Morgan fingerprint memory:", fp_array.nbytes / 1024**3, "GB")


In [ ]:
def klifs_map(df,KLIFS_file,KLIFS_latent): # df is the input table; KLIFS_file contains reference KLIFS sequences; KLIFS_latent contains their reduced features
    # Read KLIFS sequences from the input dataset
    read_klifs = df['KLIFS'].astype(str).str.strip()
    # Read KLIFS sequences from the reference dataset
    KLIFS = KLIFS_file['KLIFS'].astype(str).str.strip()
    # Build a dictionary mapping KLIFS sequences to their reduced features
    keys = KLIFS
    values = KLIFS_latent
    KLIFS_dict = dict(zip(keys,values))
    # Map the sequences
    kinase_result = [KLIFS_dict.get(x.lstrip(),None) for x in read_klifs]
    return kinase_result

In [ ]:
result = klifs_map(warm_train, KLIFS_file, KLIFS_sae_top)
none_mask = [x is None for x in result]
none_rows = warm_train[none_mask]
print(f"None count: {sum(none_mask)}")
print(none_rows[['KLIFS']].head(20))

In [ ]:
# Concatenate molecular Morgan fingerprints and kinase latent features
# Use NumPy throughout to avoid copying large DataFrames with pd.concat

kinase_array = np.asarray(klifs_map(warm_train, KLIFS_file, KLIFS_sae_top), dtype=np.float32)

# Check for unsuccessful KLIFS mappings
if np.any(pd.isna(kinase_array)):
    raise ValueError("kinase_array contains NaN, indicating unmapped KLIFS entries. Check none_rows first.")

# Convert once to float32 for XGBoost to avoid implicit float64 conversions and repeated copies
X_train = np.concatenate(
    [fp_array.astype(np.float32, copy=False), kinase_array],
    axis=1
)

y_train = np.asarray(warm_bio, dtype=np.int8)

assert X_train.shape[0] == len(y_train), "X_train and y_train have different sample counts"

print("X_train shape:", X_train.shape)
print("X_train dtype:", X_train.dtype)
print("X_train memory:", X_train.nbytes / 1024**3, "GB")


In [ ]:
print("warm_train shape:", warm_train.shape)
print("fp_array shape:", fp_array.shape)
print("kinase_array shape:", kinase_array.shape)
print("X_train shape:", X_train.shape)
print("positive ratio:", y_train.mean())


In [ ]:
# Delete intermediate objects to release memory
del warm_smiles, warm_train, warm_klifs, KLIFS_sae_top, KLIFS_file, valid_smiles, fp_array, kinase_array
gc.collect()


In [ ]:
# Do not create a full DMatrix before Optuna
# DMatrix holds an additional internal copy; slicing it during CV further increases peak memory
# Retain only the NumPy X_train and y_train arrays here

def make_dmatrix(X_part, y_part=None):
    if y_part is None:
        return xgb.DMatrix(X_part)
    return xgb.DMatrix(X_part, label=y_part)


In [ ]:
# X_train and y_train were constructed above
# Do not run: Xy_train, X_train, y_train = dataset(...)
print("Ready for Optuna:", X_train.shape, X_train.dtype, y_train.shape, y_train.dtype)


In [ ]:
gc.collect()


In [ ]:
import numpy as np

def calculate_bedroc(y_true, y_score, bedroc_alpha):
    y_true  = np.asarray(y_true)
    y_score = np.asarray(y_score)
    N = len(y_true)
    n_act = np.sum(y_true == 1)
    if n_act == 0:
        return 0.0                    # Return 0 when there are no active compounds

    # 1. Sort predictions and obtain the ranks of active compounds
    order  = np.argsort(y_score)[::-1]
    ranks  = np.arange(1, N+1)[y_true[order] == 1]      # 1-based
    x      = (ranks - 0.5) / N                          # Rank midpoint

    # 2. Calculate RIE
    C      = (1.0 - np.exp(-bedroc_alpha)) / bedroc_alpha
    rie    = np.mean(np.exp(-bedroc_alpha * x)) / C

    # 3. Calculate the limits
    x_max  = (np.arange(1, n_act+1) - 0.5) / N          # Best case
    x_min  = (np.arange(N-n_act+1, N+1) - 0.5) / N      # Worst case
    rie_max = np.mean(np.exp(-bedroc_alpha * x_max)) / C
    rie_min = np.mean(np.exp(-bedroc_alpha * x_min)) / C

    # 4. Normalize
    return (rie - rie_min) / (rie_max - rie_min)

In [ ]:
def calculate_ef(y_true, y_score, percentages=[0.5, 1.0, 2.0, 3.0,4.0, 5.0]):
    """
    Calculate the Enrichment Factor (EF).

    Args:
        y_true: Binary labels (0 or 1; 1 indicates an active compound).
        y_score: Prediction scores; higher values indicate greater predicted activity.
        percentages: Percentages at which EF is calculated; defaults to [0.5, 1.0, 5.0].

    Returns:
        A dictionary mapping each percentage to its EF.
    """
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    
    # Validate the inputs
    if len(y_true) != len(y_score):
        raise ValueError("y_true and y_score must have equal lengths")
    
    N = len(y_true)  # Total number of compounds
    n_active = np.sum(y_true == 1)  # Total number of active compounds
    
    if n_active == 0:
        print("Warning: no active compounds; returning EF=0")
        return {f"EF_{p}%": 0.0 for p in percentages}
    
    # Sort by descending prediction score
    sorted_indices = np.argsort(y_score)[::-1]
    sorted_labels = y_true[sorted_indices]
    
    results = {}
    
    for percentage in percentages:
        # Calculate the number of compounds in the top x%
        n_selected = max(1, int(N * percentage / 100))  # Select at least one compound
        
        # Count active compounds in the selected subset
        n_active_in_selected = np.sum(sorted_labels[:n_selected])
        
        # Calculate the enrichment factor
        # EF = active fraction in the top x% / active fraction in the full dataset
        ef_value = (n_active_in_selected / n_selected) / (n_active / N)
        
        results[f"EF_{percentage}%"] = ef_value
        
        # Print detailed results
        print(f"EF_{percentage}%:")
        print(f"  - Selected compounds: {n_selected}/{N} ({percentage}%)")
        print(f"  - Active compounds among selected: {n_active_in_selected}")
        print(f"  - Fraction of active compounds among selected: {n_active_in_selected/n_selected:.4f}")
        print(f"  - Overall fraction of active compounds: {n_active/N:.4f}")
        print(f"  - EF value: {ef_value:.4f}")
        print()
    
    return results

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

def live_plot_callback(study, trial):
    # Use a lightweight plotting callback to limit overhead during large hyperparameter searches
    clear_output(wait=True)

    completed = [t for t in study.trials
                 if t.state == optuna.trial.TrialState.COMPLETE]

    trial_numbers = [t.number for t in completed]
    auc_vals      = [t.user_attrs.get("roc_auc",  np.nan) for t in completed]
    bedroc_vals   = [t.user_attrs.get("bedroc",   np.nan) for t in completed]

    fig = plt.figure(figsize=(8, 4))
    plt.plot(trial_numbers, auc_vals,    marker='o', label='ROC-AUC')
    plt.plot(trial_numbers, bedroc_vals, marker='x', label='BEDROC')
    plt.xlabel("Trial")
    plt.ylabel("Score")
    plt.ylim(0, 1)
    plt.title("Optuna live progress")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()
    plt.close(fig)


In [ ]:
def objective(trial, X, y, n_folds, bedroc_alpha):

    params = {
        "tree_method": "hist",
        "device": "cuda",          # Set the device to "cpu" if GPU memory is insufficient
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "random_state": 42,

        # Consider a maximum depth of 8; depth 10 uses substantial memory for 690k x 4200 inputs
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 5, 10),
        "subsample": trial.suggest_float("subsample", 0.3, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 0.8),
        "lambda": trial.suggest_float("lambda", 1e-3, 1e3, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 300, log=True),
        "gamma": trial.suggest_float("gamma", 0, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 12),
        "scale_pos_weight": 14,
        "nthread": 4,
    }

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

    fold_auc = []
    fold_bedroc = []
    best_iters = []

    for fold_id, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        # Fold subsets create copies; release them immediately after use
        # Avoid retaining both the full DMatrix and sliced fold matrices
        X_tr = X[train_idx]
        y_tr = y[train_idx]
        X_va = X[valid_idx]
        y_va = y[valid_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_va, label=y_va)

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=10000,      # Use early stopping to limit the number of boosting rounds
            evals=[(dval, "eval")],
            early_stopping_rounds=50,
            verbose_eval=False,
        )

        best_iter = model.best_iteration + 1
        best_iters.append(best_iter)

        y_pred = model.predict(dval, iteration_range=(0, best_iter))

        auc = roc_auc_score(y_va, y_pred)
        bedroc = calculate_bedroc(y_va, y_pred, bedroc_alpha=bedroc_alpha)

        fold_auc.append(auc)
        fold_bedroc.append(bedroc)

        # Prune poor Optuna trials early to reduce training time and memory use
        trial.report(float(np.mean(fold_auc)), step=fold_id)
        if trial.should_prune():
            del X_tr, X_va, y_tr, y_va, dtrain, dval, model, y_pred
            gc.collect()
            raise optuna.TrialPruned()

        # Release fold-specific objects immediately after each fold
        del X_tr, X_va, y_tr, y_va, dtrain, dval, model, y_pred
        gc.collect()

    avg_auc = float(np.mean(fold_auc))
    avg_bedroc = float(np.mean(fold_bedroc))
    avg_best_iter = int(np.round(np.mean(best_iters)))

    trial.set_user_attr("avg_best_iter", avg_best_iter)
    trial.set_user_attr("roc_auc", avg_auc)
    trial.set_user_attr("bedroc", avg_bedroc)

    print(f"[Validation] BEDROC alpha = {bedroc_alpha}")
    print(f"avg best iteration = {avg_best_iter}")
    print(f"avg AUC = {avg_auc:.4f}, avg BEDROC = {avg_bedroc:.4f}")

    # Return avg_bedroc instead if BEDROC is the optimization objective
    return avg_auc


In [ ]:
# Start Optuna Bayesian optimization
# Check stability with three folds and 30 trials before increasing folds or trials
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
)

study.optimize(
    partial(objective, X=X_train, y=y_train, n_folds=5, bedroc_alpha=80.5),
    n_trials=200,
    callbacks=[live_plot_callback],
    show_progress_bar=True,
    gc_after_trial=True,
)

best_params = study.best_params
best_iter = study.best_trial.user_attrs["avg_best_iter"]

print("Best hyperparameters:", best_params)
print("Best iteration:", best_iter)


: 

In [ ]:
final_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "device": "cuda",   # Set the device to "cpu" if GPU memory is insufficient
    "scale_pos_weight": 14,
    **best_params,
}

dtrain_full = xgb.DMatrix(X_train, label=y_train)

final_model = xgb.train(
    final_params,
    dtrain_full,
    num_boost_round=best_iter,
)

final_model.save_model(
    _kp('model/Morgan_raw/xgboost_SPECTRA_ready_to_use_with_VSDS_482_kinases_4096bit_morgan_2.json')
)

del dtrain_full
gc.collect()


In [ ]:
# Retain final_model if it will be used for VSDS predictions
# del X_train, y_train
gc.collect()


In [ ]:
# Define a function to evaluate the trained model
def run_the_model_full(Xy,y,kinase_name):

    # Load the model
    final_model = xgb.Booster()
    final_model.load_model(_kp('model/Morgan_raw/xgboost_SPECTRA_ready_to_use_with_VSDS_482_kinases_4096bit_morgan_2.json'))
    y_pred_prob = final_model.predict(Xy)  # Prediction probabilities

    # Select the threshold that maximizes F1
    precisions, recalls, thresholds = precision_recall_curve(y, y_pred_prob)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    optimal_idx = np.argmax(f1_scores)
    y_pred = (y_pred_prob > thresholds[optimal_idx])

    # Calculate the evaluation metrics
    auc = roc_auc_score(y, y_pred_prob)
    auc_var = np.var(y_pred_prob) / len(y_pred_prob)
    z = norm.ppf(1 - (1 - 0.95) / 2)  # Calculate the z-score
    lower_bound = auc - z * np.sqrt(auc_var)
    upper_bound = auc + z * np.sqrt(auc_var)

    acc = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    mcc = matthews_corrcoef(y, y_pred)
    logloss = log_loss(y, y_pred_prob)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) != 0 else 0.0

     # Calculate confidence intervals by bootstrapping
    n_bootstraps = 1000
    bootstrapped_aucs = []

    # Convert y and y_pred_prob to NumPy arrays
    y = np.array(y)
    y_pred_prob = np.array(y_pred_prob)

    rng = np.random.RandomState(42)
    for _ in range(n_bootstraps):
        indices = rng.choice(len(y), size=len(y), replace=True)
        auc_bootstrap = roc_auc_score(y[indices], y_pred_prob[indices])
        bootstrapped_aucs.append(auc_bootstrap)
    ci_lower = np.percentile(bootstrapped_aucs, 2.5)
    ci_upper = np.percentile(bootstrapped_aucs, 97.5)
    print(f"AUC: {auc:.3f} (95% CI: [{ci_lower:.3f}, {ci_upper:.3f}])")

    # Calculate BEDROC
    bedroc_score = calculate_bedroc(y, y_pred_prob, bedroc_alpha=80.5)
    ef_score = calculate_ef(y, y_pred_prob)
    # Print results
    print(f"Best threshold: {thresholds[optimal_idx]:.4f}")
    print(f"AUC: {auc:.4f} (95% CI: {lower_bound:.3f}, {upper_bound:.3f})")
    print(f"bedroc: {bedroc_score:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision(PPV): {precision:.4f}")
    print(f"Recall(Sensitivity): {recall:.4f}")
    print(f"F1 score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"NPV: {npv:.4f}")
    print(f"MCC: {mcc:.4f}")
    print(f"Log Loss: {logloss:.4f}")
    print(confusion_matrix(y, y_pred))
    print(f"True Negative = {tn}")
    print(f"True Positive = {tp}")
    print(f"False Negative = {fn}")
    print(f"False Positive = {fp}")
    print("Class distribution:", Counter(y))
    print("Positive-class fraction:", np.mean(y))

    if kinase_name is not None:
        df_output = pd.DataFrame({
            "Best threshold": f"{thresholds[optimal_idx]:.4f}",
            "AUC": f"{auc:.4f}",
            "AUC 95% CI lower_bound": f"{lower_bound:.3f}",
            "AUC 95% CI upper_bound": f"{upper_bound:.3f}",
            "bedroc": f"{bedroc_score:.4f}",
            "Accuracy": f"{acc:.4f}",
            "Precision(PPV)": f"{precision:.4f}",
            "Recall(Sensitivity)": f"{recall:.4f}",
            "F1 score": f"{f1:.4f}",
            "Specificity": f"{specificity:.4f}",
            "NPV": f"{npv:.4f}",
            "MCC": f"{mcc:.4f}",
            "Log Loss": f"{logloss:.4f}",
            "True Negative": f"{tn}",
            "True Positive": f"{tp}",
            "False Negative": f"{fn}",
            "False Positive": f"{fp}"
        },index=[kinase_name])
        return bedroc_score,df_output,ef_score
    else:
        return bedroc_score

In [ ]:
vsds = pd.read_csv(_kp('dataset/VSDS_truedecoy_integrated_data_with_protein_seq.csv'))
vsds_smiles = vsds["SMILES"]

vsds_valid_smiles = []
vsds_fp_array = np.zeros((len(vsds_smiles), 4096), dtype=np.uint8)

vsds_valid_idx = 0

for vsds_smi in vsds_smiles:
    vsds_mol = Chem.MolFromSmiles(vsds_smi)
    if vsds_mol is None:
        print(f"[Warning] Unable to parse SMILES: {vsds_smi}")
        continue

    vsds_fp = mfpgen.GetFingerprint(vsds_mol)
    ConvertToNumpyArray(vsds_fp, vsds_fp_array[vsds_valid_idx])
    vsds_valid_smiles.append(vsds_smi)
    vsds_valid_idx += 1

vsds_fp_array = vsds_fp_array[:vsds_valid_idx]

print("VSDS Morgan shape:", vsds_fp_array.shape)


In [ ]:
vsds_klifs = vsds["KLIFS"]
vsds_bio = np.asarray(vsds["molecule_label"], dtype=np.int8)

KLIFS_sae_top = pd.read_csv(
    _kp('feature_embedding/Kinase_test_MCA/MCA_klifs_482_200D_features.csv'),
    skiprows=1,
    header=None
).values

KLIFS_file = pd.read_csv(
    _kp('feature_embedding/Kinase_test_MCA/human_482_typical_kinase_1955bit.csv')
)

vsds_kinase_array = np.asarray(klifs_map(vsds, KLIFS_file, KLIFS_sae_top), dtype=np.float32)

vsds_X = np.concatenate(
    [vsds_fp_array.astype(np.float32, copy=False), vsds_kinase_array],
    axis=1
)

print("VSDS X shape:", vsds_X.shape)
print("VSDS X memory:", vsds_X.nbytes / 1024**3, "GB")

vsds_Xy = xgb.DMatrix(vsds_X, label=vsds_bio)


In [ ]:
print('evaluation metrics on VSDS test set:')
vsds_test = run_the_model_full(vsds_Xy,vsds_bio,None)